In [2]:
import os
import sys

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))
import ast
import re

import nltk
import numpy as np
import pandas as pd
# sys.path.append('/ocean/projects/cis250068p/jhwang4/idl-project')
import torch as t
import torch.nn.functional as F
from datasets import Dataset, load_dataset
from nltk.tokenize import sent_tokenize
from nltk.translate.bleu_score import SmoothingFunction, sentence_bleu
from rouge_score import rouge_scorer
from scipy.spatial.distance import cosine
from sentence_transformers import SentenceTransformer, util
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from tqdm.auto import tqdm
from transformers import AutoTokenizer, GPTNeoXForCausalLM

from src.data import load_cnn_dataset
from src.model import *
from src.search_batch_v2 import *
from src.utils_batch_v2 import *

In [3]:
pd.set_option('display.max_colwidth', 200)
pd.set_option('display.float_format', '{:.4f}'.format)
nltk.download("punkt_tab")


[nltk_data] Downloading package punkt_tab to
[nltk_data]     /jet/home/iwiryadi/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [4]:
DEVICE

'cuda'

In [5]:
# Load the models
fo_model, fo_tokenizer = load_ftfo_model()
ba_model, ba_tokenizer = load_ftba_model()
fo_model = fo_model.to(DEVICE)
ba_model = ba_model.to(DEVICE)
fo_model.half()
ba_model.half()

# 멀티-GPU 자동 적용
if t.cuda.device_count() > 1:
    print(t.cuda.device_count())
    fo_model = t.nn.DataParallel(fo_model)
    ba_model = t.nn.DataParallel(ba_model)

4


In [6]:
CACHE_DIR += 'iw'

In [7]:
# dataset = load_cnn_dataset(num_samples=95704, cache_dir=CACHE_DIR)
dataset = load_cnn_dataset(num_samples=3200, cache_dir=CACHE_DIR)

dataset = pd.DataFrame(dataset)

Dataset loaded successfully
Example dataset item: {'article': 'LONDON, England (Reuters) -- Harry Potter star Daniel Radcliffe gains access to a reported £20 million ($41.1 million) fortune as he turns 18 on Monday, but he insists the money won\'t cast a spell on him. Daniel Radcliffe as Harry Potter in "Harry Potter and the Order of the Phoenix" To the disappointment of gossip columnists around the world, the young actor says he has no plans to fritter his cash away on fast cars, drink and celebrity parties. "I don\'t plan to be one of those people who, as soon as they turn 18, suddenly buy themselves a massive sports car collection or something similar," he told an Australian interviewer earlier this month. "I don\'t think I\'ll be particularly extravagant. "The things I like buying are things that cost about 10 pounds -- books and CDs and DVDs." At 18, Radcliffe will be able to gamble in a casino, buy a drink in a pub or see the horror film "Hostel: Part II," currently six places be

In [8]:
binary_results = binary_search_attribution_search(dataset, fo_model, fo_tokenizer, ba_model, ba_tokenizer, max_iterations=30, sentence_batch_size=500) # 35.7s, 7m 47.6s, 5m 15.6s

  0%|          | 0/3200 [00:00<?, ?it/s]

/ocean/projects/cis250068p/iwiryadi/idl2/idl-project/src/utils_batch_v2.py:245: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with t.no_grad(), autocast():


In [9]:
EMB_MODEL    = SentenceTransformer('all-MiniLM-L6-v2')
ROUGE_SCORER = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
TFIDF_VEC    = TfidfVectorizer(
    token_pattern=r"(?u)\b\w+\b",  # 좀 더 느슨한 토큰화
    stop_words=None               # 불용어 필터링 끄기
)

def calculate_embedding_similarity(highlight, citation):
    he = EMB_MODEL.encode([highlight])[0]
    ce = EMB_MODEL.encode([citation])[0]
    return 1 - cosine(he, ce)

def calculate_rouge_score(highlight, citation):
    return ROUGE_SCORER.score(highlight, citation)['rougeL'].fmeasure

def calculate_tfidf_score(highlight, citation):
    try:
        mat = TFIDF_VEC.fit_transform([highlight, citation]).toarray()
        return 1 - cosine(mat[0], mat[1])
    except ValueError:
        # 어휘가 하나도 없으면 0.0
        return 0.0

def process_data(data):
    results = []
    for item in data:
        hl = item['highlight']
        rec = {'id': item['id'], 'highlight': hl}
        for citation_type in ['base_citation','fo_citation','ba_citation']:
            prefix = citation_type.split('_')[0]
            cit = item.get(citation_type, None)
            if cit:
                rec[f'{prefix}_emb_similarity'] = calculate_embedding_similarity(hl, cit)
                rec[f'{prefix}_rouge_score']     = calculate_rouge_score(hl, cit)
                rec[f'{prefix}_tfidf_score']     = calculate_tfidf_score(hl, cit)
            else:
                rec[f'{prefix}_emb_similarity'] = None
                rec[f'{prefix}_rouge_score']     = None
                rec[f'{prefix}_tfidf_score']     = None
        results.append(rec)
    return results

binary_final_results    = process_data(binary_results)

In [10]:
r = pd.DataFrame(binary_final_results) 
r.drop(['id', 'highlight'], axis=1).mean()

base_emb_similarity   0.4926
base_rouge_score      0.2064
base_tfidf_score      0.1809
fo_emb_similarity     0.6491
fo_rouge_score        0.3354
fo_tfidf_score        0.2901
ba_emb_similarity     0.6697
ba_rouge_score        0.3460
ba_tfidf_score        0.3016
dtype: float64

In [11]:
df_binary   = pd.DataFrame(binary_final_results)
mean_series = df_binary.drop(['id','highlight'],axis=1).mean()

data = {
    'Base_binary': [
        mean_series['base_emb_similarity'],
        mean_series['base_rouge_score'],
        mean_series['base_tfidf_score']
    ],
    'Fo_binary': [
        mean_series['fo_emb_similarity'],
        mean_series['fo_rouge_score'],
        mean_series['fo_tfidf_score']
    ],
    'Ba_binary': [
        mean_series['ba_emb_similarity'],
        mean_series['ba_rouge_score'],
        mean_series['ba_tfidf_score']
    ],
}

table_df = pd.DataFrame(data, index=['Embedding','Rouge','Tfidf'])
display(table_df)
#table_df
table_df.to_csv("output/ft_binary_result.csv", index=True)

,Base_binary,Fo_binary,Ba_binary
Embedding,0.4926,0.6491,0.6697
Rouge,0.2064,0.3354,0.3460
Tfidf,0.1809,0.2901,0.3016
